In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import root_mean_squared_error


In [ ]:
df = pd.read_csv("../../data/FRB_H15.csv").dropna()

#rename columns
df.rename(columns={"Series Description": "Date", "Market yield on U.S. Treasury securities at 1-month  constant maturity, quoted on investment basis": "0Y1M", "Market yield on U.S. Treasury securities at 3-month  constant maturity, quoted on investment basis": "0Y3M", "Market yield on U.S. Treasury securities at 6-month  constant maturity, quoted on investment basis": "0Y6M", "Market yield on U.S. Treasury securities at 1-year  constant maturity, quoted on investment basis": "1Y", "Market yield on U.S. Treasury securities at 2-year  constant maturity, quoted on investment basis": "2Y", "Market yield on U.S. Treasury securities at 3-year  constant maturity, quoted on investment basis": "3Y", "Market yield on U.S. Treasury securities at 5-year  constant maturity, quoted on investment basis": "5Y", "Market yield on U.S. Treasury securities at 7-year  constant maturity, quoted on investment basis": "7Y", "Market yield on U.S. Treasury securities at 10-year  constant maturity, quoted on investment basis": "10Y", "Market yield on U.S. Treasury securities at 20-year constant maturity, quoted on investment basis": "20Y", "Market yield on U.S. Treasury securities at 30-year  constant maturity, quoted on investment basis": "30Y"}, inplace=True)
df.rename(columns={"Market yield on U.S. Treasury securities at 20-year  constant maturity, quoted on investment basis": "20Y"}, inplace=True)

#make index to datetime for timeseries
#df["Date"] = pd.to_datetime(df["Date"])
# dont make timeseries data for xgboost

#df.set_index("Date", inplace=True)
df = df.reset_index(drop=True)
df.head(6)

In [ ]:
maturities = []
matList = ["0Y1M", "0Y3M", "0Y6M", "1Y", "2Y", "3Y", "5Y", "7Y", "10Y", "20Y", "30Y"]

for col in df.columns:
    maturities.append(df[col])

# 1D table array, the given index is the same index in matlist
tables = []

#lag features for all maturities, 5 lags for each maturity, 55 features in total
features = [
    "0Y1M_1", "0Y1M_2", "0Y1M_3", "0Y1M_4", "0Y1M_5",
    "0Y3M_1", "0Y3M_2", "0Y3M_3", "0Y3M_4", "0Y3M_5",
    "0Y6M_1", "0Y6M_2", "0Y6M_3", "0Y6M_4", "0Y6M_5",
    "1Y_1", "1Y_2", "1Y_3", "1Y_4", "1Y_5",
    "2Y_1", "2Y_2", "2Y_3", "2Y_4", "2Y_5",
    "3Y_1", "3Y_2", "3Y_3", "3Y_4", "3Y_5",
    "5Y_1", "5Y_2", "5Y_3", "5Y_4", "5Y_5",
    "7Y_1", "7Y_2", "7Y_3", "7Y_4", "7Y_5",
    "10Y_1", "10Y_2", "10Y_3", "10Y_4", "10Y_5",
    "20Y_1", "20Y_2", "20Y_3", "20Y_4", "20Y_5",
    "30Y_1", "30Y_2", "30Y_3", "30Y_4", "30Y_5",
    "target"    
]


for mat in matList:

    #arrays for data in each horizon, so rows1 has table with 
    rows1 = []
    rows2 = []
    rows3 = []



    for i in range(4, len(df.index)-1):
        target = df[mat][i+1]

        lagAll = []
        for mat in matList:
            lag1 = df[mat][i-1]
            lag2 = df[mat][i-2]
            lag3 = df[mat][i-3]
            lag4 = df[mat][i-4]
            lag5 = df[mat][i-5]
            lagAll.extend([lag1, lag2, lag3, lag4, lag5])
        lagAll.extend([target])

        #adds a row with 5 lags for each maturity, and then target value for the specific maturity that we are creating data for
        rows1.append(lagAll)

    for i in range(4, len(df.index)-5):
        target = df[mat][i+5]

        lagAll = []
        for mat in matList:
            lag1 = df[mat][i-1]
            lag2 = df[mat][i-2]
            lag3 = df[mat][i-3]
            lag4 = df[mat][i-4]
            lag5 = df[mat][i-5]
            lagAll.extend([lag1, lag2, lag3, lag4, lag5])
        lagAll.extend([target])

        #adds a row with 5 lags for each maturity, and then target value for the specific maturity that we are creating data for
        rows2.append(lagAll)
    for i in range(4, len(df.index)-20):
        target = df[mat][i+20]

        lagAll = []
        for mat in matList:
            lag1 = df[mat][i-1]
            lag2 = df[mat][i-2]
            lag3 = df[mat][i-3]
            lag4 = df[mat][i-4]
            lag5 = df[mat][i-5]
            lagAll.extend([lag1, lag2, lag3, lag4, lag5])
        lagAll.extend([target])

        #adds a row with 5 lags for each maturity, and then target value for the specific maturity that we are creating data for
        rows3.append(lagAll)
    
    df1 = pd.DataFrame(rows1, columns=features)
    df2 = pd.DataFrame(rows2, columns=features)
    df3 = pd.DataFrame(rows3, columns=features)


    tables.append([df1, df2, df3])
        
#now tables[i] gives the ith maturity, and tables[i][0] gives the 1-day ahead table
# , tables[i][1] gives the 5-day ahead table, and tables[i][2] gives the 20-day ahead table
        